# High-Energy Physics with Python

This notebook introduces a small set of HEP-specific Python tools through a simplified analysis workflow:

- `particle` for particle properties and PDG identifiers
- `fastjet` for jet clustering
- `iminuit` for parameter estimation
- `pyhf` for statistical inference

The examples build on the Scientific Python stack, especially NumPy, Matplotlib, Hist, and mplhep.

## Learning objectives

By the end of this notebook, you will be able to:

- Look up particle properties with `particle`
- Represent simple collision events as structured NumPy arrays
- Cluster final-state particles into jets with `fastjet`
- Visualize physics distributions with HEP-style plotting tools
- Fit model parameters with `iminuit`
- Build a simple likelihood model and perform inference with `pyhf`
- Understand how these packages fit into a typical HEP analysis pipeline

## The HEP Python ecosystem

```text
Experimental data
      │
      ▼
Uproot / Parquet
      │
      ▼
Awkward Array
      │
      ├──────────────┬──────────────┐
      ▼              ▼              ▼
   Vector          Hist          Particle
      │              │              │
      └──────────────┴──────────────┘
                     │
                     ▼
               Physics analysis
                     │
          ┌──────────┴──────────┐
          ▼                     ▼
       fastjet               iminuit
   Jet clustering       Parameter fitting
          │                     │
          └──────────┬──────────┘
                     ▼
                    pyhf
          Statistical inference
```

## Setup

Install the required packages before running the notebook:

```bash
pip install numpy matplotlib hist mplhep particle iminuit pyhf pyjet
```

Some environments may require a compiler or prebuilt wheel for `pyjet`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from particle import Particle
from iminuit import Minuit
from iminuit.cost import LeastSquares

import pyhf
import fastjet

from hist import Hist
import hist
import mplhep as hep

# 1. Particle properties with `particle`

The `particle` package provides access to particle properties using names or PDG identifiers.

This avoids hard-coding masses, charges, lifetimes, and identifiers throughout an analysis.

In [ ]:
muon = Particle.from_pdgid(13)

print(muon)
print("Name:", muon.name)
print("PDG ID:", muon.pdgid)
print("Mass [MeV]:", muon.mass)
print("Charge [e]:", muon.charge)
print("Lifetime [s]:", muon.lifetime)

You can also look up particles by name:

In [ ]:
higgs = Particle.from_name("H0")
top = Particle.from_name("t")

print("Higgs mass [MeV]:", higgs.mass)
print("Top-quark charge [e]:", top.charge)

## Exercise 1

Use `particle` to find:

1. The PDG ID of the electron
2. The mass of the charged pion
3. The charge of the anti-muon
4. The lifetime of the tau lepton

In [ ]:
# Your solution here

electron = ...
charged_pion = ...
antimuon = ...
tau = ...

# 2. Jet clustering with `fastjet`

Quarks and gluons are not observed directly. Instead, they fragment and hadronize into collimated sprays of particles called **jets**.

Jet-clustering algorithms group nearby particles into jet candidates. A common choice is the anti-$k_t$ algorithm, which tends to produce approximately cone-shaped jets.

The array-oriented `fastjet` interface accepts an Awkward Array containing particle four-momenta. Records with Cartesian fields

* `px`
* `py`
* `pz`
* `E`

are recognized directly.

Alternatively, particles may be represented using

* `pt`
* `eta`
* `phi`
* `M`

when Vector behavior is registered and the records are given the name `Momentum4D`.

The outer array represents events, while each event contains a variable-length collection of particles.

In [ ]:
import awkward as ak
import fastjet
import vector

vector.register_awkward()

particles = ak.Array(
    [
        [
            {"pt": 50.0, "eta": 0.10, "phi": 0.05, "M": 0.0},
            {"pt": 35.0, "eta": 0.12, "phi": 0.08, "M": 0.0},
            {"pt": 20.0, "eta": -1.20, "phi": 2.80, "M": 0.0},
        ],
        [
            {"pt": 15.0, "eta": -1.18, "phi": 2.75, "M": 0.0},
            {"pt": 5.0, "eta": 0.50, "phi": -2.40, "M": 0.0},
        ],
    ],
    with_name="Momentum4D",
)

jet_definition = fastjet.JetDefinition(
    fastjet.antikt_algorithm,
    0.4,
)

cluster = fastjet.ClusterSequence(particles, jet_definition)
jets = cluster.inclusive_jets(min_pt=10.0)

jets

In [ ]:
if fastjet is None:
    print("fastjet is not installed in this environment.")
else:
    jets = cluster.inclusive_jets(min_pt=10.0)

    for event, event_jets in enumerate(jets):
        print(f"Event {event}")

        for i, jet in enumerate(event_jets):
            print(
                f"  Jet {i}: "
                f"pT={jet.pt:.2f} GeV, "
                f"eta={jet.eta:.2f}, "
                f"phi={jet.phi:.2f}, "
                f"mass={jet.mass:.2f} GeV"
            )

The `JetDefinition` specifies both the clustering algorithm and its radius parameter `R`.

Common choices include:

* `fastjet.antikt_algorithm` — anti-$k_t$ (the standard algorithm used by ATLAS and CMS)
* `fastjet.cambridge_algorithm` — Cambridge/Aachen
* `fastjet.kt_algorithm` — $k_t$

For example:

```python
jet_definition = fastjet.JetDefinition(
    fastjet.antikt_algorithm,
    R=0.4,
)
```

The radius parameter `R` controls the characteristic size of the reconstructed jets. Smaller values produce narrower jets, while larger values merge particles over a wider angular region.

## Exercise 2

Using the same particle collection:

1. Cluster the particles with the anti-$k_t$ algorithm using $R=0.6$.
2. Compare the number of jets with the anti-$k_t$, $R=0.4$ result.
3. Repeat the clustering with the $k_t$ algorithm using $R=0.4$.
4. Compare the jet transverse momenta produced by the different configurations.

Try to keep the analysis array-oriented and avoid converting the jets into Python lists.

<details>
<summary>Solution</summary>

```python
import awkward as ak
import fastjet

# Anti-k_t with R = 0.4
antikt_R04 = fastjet.JetDefinition(
    fastjet.antikt_algorithm,
    0.4,
)

cluster_antikt_R04 = fastjet.ClusterSequence(
    particles,
    antikt_R04,
)

jets_antikt_R04 = cluster_antikt_R04.inclusive_jets(
    min_pt=10.0,
)

# Anti-k_t with R = 0.6
antikt_R06 = fastjet.JetDefinition(
    fastjet.antikt_algorithm,
    0.6,
)

cluster_antikt_R06 = fastjet.ClusterSequence(
    particles,
    antikt_R06,
)

jets_antikt_R06 = cluster_antikt_R06.inclusive_jets(
    min_pt=10.0,
)

# k_t with R = 0.4
kt_R04 = fastjet.JetDefinition(
    fastjet.kt_algorithm,
    0.4,
)

cluster_kt_R04 = fastjet.ClusterSequence(
    particles,
    kt_R04,
)

jets_kt_R04 = cluster_kt_R04.inclusive_jets(
    min_pt=10.0,
)

# Count the number of jets in each event.
n_jets_antikt_R04 = ak.num(jets_antikt_R04, axis=1)
n_jets_antikt_R06 = ak.num(jets_antikt_R06, axis=1)
n_jets_kt_R04 = ak.num(jets_kt_R04, axis=1)

print("Number of jets per event")
print("anti-k_t, R=0.4:", n_jets_antikt_R04.tolist())
print("anti-k_t, R=0.6:", n_jets_antikt_R06.tolist())
print("k_t,      R=0.4:", n_jets_kt_R04.tolist())

print()
print("Jet pT values per event")
print("anti-k_t, R=0.4:", jets_antikt_R04.pt.tolist())
print("anti-k_t, R=0.6:", jets_antikt_R06.pt.tolist())
print("k_t,      R=0.4:", jets_kt_R04.pt.tolist())
```

A larger radius parameter allows particles separated by a wider angular distance to be clustered into the same jet. As a result, increasing $R$ can reduce the number of reconstructed jets and increase the transverse momentum of the merged jets.

The anti-$k_t$ and $k_t$ algorithms use different clustering orderings, so they can produce different jet assignments even when the same radius parameter is used.

</details>

In [ ]:
# Your solution here

# 3. Histogramming HEP observables

Histogramming is central to HEP analysis. We often compare observed distributions against simulated signal and background predictions.

The following example generates a toy invariant-mass distribution.

In [ ]:
rng = np.random.default_rng(42)

background = rng.exponential(scale=35.0, size=20_000) + 50.0
signal = rng.normal(loc=125.0, scale=2.0, size=1_000)

mass = np.concatenate([background, signal])
mass = mass[(mass > 80.0) & (mass < 170.0)]

mass[:10]

In [ ]:
if Hist is not None:
    h_mass = (
        Hist.new
        .Reg(45, 80.0, 170.0, name="mass", label=r"$m$ [GeV]")
        .Double()
    )
    h_mass.fill(mass=mass)

    if hep is not None:
        plt.style.use(hep.style.CMS)
        hep.histplot(h_mass)
    else:
        h_mass.plot()

    plt.xlabel(r"Invariant mass [GeV]")
    plt.ylabel("Events")
    plt.title("Toy resonance search")
    plt.show()
else:
    plt.hist(mass, bins=45, range=(80.0, 170.0), histtype="step")
    plt.xlabel(r"Invariant mass [GeV]")
    plt.ylabel("Events")
    plt.title("Toy resonance search")
    plt.show()

# 4. Parameter estimation with `iminuit`

`iminuit` provides a Python interface to the Minuit numerical minimization library.

A common fitting workflow is:

1. Define a model
2. Define a cost function
3. Initialize `Minuit`
4. Run `migrad`
5. Estimate uncertainties using `hesse` or `minos`

## Linear fit example

In [ ]:
rng = np.random.default_rng(7)

x = np.linspace(0.0, 10.0, 30)
y_true = 2.5 * x + 1.2
y_err = np.full_like(x, 1.5)
y = rng.normal(y_true, y_err)

def linear_model(x, slope, intercept):
    return slope * x + intercept

least_squares = LeastSquares(x, y, y_err, linear_model)

fit = Minuit(
    least_squares,
    slope=1.0,
    intercept=0.0,
)

fit.migrad()
fit.hesse()

fit

In [ ]:
print("Best-fit slope:", fit.values["slope"])
print("Slope uncertainty:", fit.errors["slope"])
print("Best-fit intercept:", fit.values["intercept"])
print("Intercept uncertainty:", fit.errors["intercept"])

In [ ]:
x_plot = np.linspace(x.min(), x.max(), 200)

plt.errorbar(x, y, yerr=y_err, fmt="o", label="Data")
plt.plot(
    x_plot,
    linear_model(
        x_plot,
        fit.values["slope"],
        fit.values["intercept"],
    ),
    label="Best fit",
)

plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

## Exercise 3

Fit a Gaussian model to the generated mass peak.

Suggested model:

```python
def gaussian(x, amplitude, mean, sigma):
    return amplitude * np.exp(-0.5 * ((x - mean) / sigma) ** 2)
```

Estimate:

* the peak position (`mean`)
* the width (`sigma`)
* the normalization (`amplitude`)

<details>
<summary>Solution</summary>

```python
import numpy as np
import matplotlib.pyplot as plt

from iminuit import Minuit
from iminuit.cost import LeastSquares

# Build a histogram of the generated mass spectrum.
counts, edges = np.histogram(
    mass,
    bins=45,
    range=(80.0, 170.0),
)

centers = 0.5 * (edges[:-1] + edges[1:])

# Use Poisson uncertainties.
errors = np.sqrt(np.maximum(counts, 1.0))


def gaussian(x, amplitude, mean, sigma):
    return amplitude * np.exp(
        -0.5 * ((x - mean) / sigma) ** 2
    )


# Construct the least-squares cost function.
least_squares = LeastSquares(
    centers,
    counts,
    errors,
    gaussian,
)

# Choose reasonable starting values.
fit = Minuit(
    least_squares,
    amplitude=900,
    mean=125.0,
    sigma=2.0,
)

fit.migrad()
fit.hesse()

print(fit)

print(f"Amplitude = {fit.values['amplitude']:.1f}")
print(f"Mean      = {fit.values['mean']:.2f} GeV")
print(f"Sigma     = {fit.values['sigma']:.2f} GeV")

# Plot the fitted model.
x = np.linspace(80, 170, 500)

plt.errorbar(
    centers,
    counts,
    yerr=errors,
    fmt="o",
    label="Data",
)

plt.plot(
    x,
    gaussian(
        x,
        fit.values["amplitude"],
        fit.values["mean"],
        fit.values["sigma"],
    ),
    lw=2,
    label="Gaussian fit",
)

plt.xlabel("Invariant mass [GeV]")
plt.ylabel("Events")
plt.legend()
plt.show()
```

**Discussion**

The fitted parameters correspond to:

* **Amplitude** — the height of the Gaussian peak.
* **Mean** — the estimated resonance mass.
* **Sigma** — the detector resolution (or intrinsic width, depending on the model).

In a realistic analysis, the signal would usually be fitted together with a background model rather than with a Gaussian alone.

</details>


In [ ]:
# Your solution here

# 5. Statistical inference with `pyhf`

`pyhf` implements HistFactory-style likelihood models in pure Python.

It can be used for:

- maximum-likelihood estimation
- hypothesis testing
- significance calculations
- confidence intervals
- upper limits

We begin with a simple one-bin counting experiment.

In [ ]:
signal = [5.0]
background = [20.0]
background_uncertainty = [4.0]
observed = [28.0]

model = pyhf.simplemodels.uncorrelated_background(
    signal=signal,
    bkg=background,
    bkg_uncertainty=background_uncertainty,
)

data = observed + model.config.auxdata

print("Parameters:", model.config.par_names)
print("Data:", data)

The parameter of interest is the signal-strength parameter $\mu$:

- $\mu = 0$: background-only hypothesis
- $\mu = 1$: nominal signal-plus-background hypothesis

In [ ]:
best_fit = pyhf.infer.mle.fit(data, model)

print("Best-fit parameters:", best_fit)
print("Best-fit signal strength:", best_fit[model.config.poi_index])

## Hypothesis test

Test the nominal signal hypothesis, $\mu=1$:

In [ ]:
cls_obs = pyhf.infer.hypotest(
    1.0,
    data,
    model,
    test_stat="qtilde",
)

print("Observed CLs:", float(cls_obs))

## Exercise 3

Fit a Gaussian model to the generated mass peak.

Suggested model:

```python
def gaussian(x, amplitude, mean, sigma):
    return amplitude * np.exp(-0.5 * ((x - mean) / sigma) ** 2)
```

Estimate:

* the peak position (`mean`)
* the width (`sigma`)
* the normalization (`amplitude`)

<details>
<summary>Solution</summary>

```python
import numpy as np
import matplotlib.pyplot as plt

from iminuit import Minuit
from iminuit.cost import LeastSquares

# Build a histogram of the generated mass spectrum.
counts, edges = np.histogram(
    mass,
    bins=45,
    range=(80.0, 170.0),
)

centers = 0.5 * (edges[:-1] + edges[1:])

# Use Poisson uncertainties.
errors = np.sqrt(np.maximum(counts, 1.0))


def gaussian(x, amplitude, mean, sigma):
    return amplitude * np.exp(
        -0.5 * ((x - mean) / sigma) ** 2
    )


# Construct the least-squares cost function.
least_squares = LeastSquares(
    centers,
    counts,
    errors,
    gaussian,
)

# Choose reasonable starting values.
fit = Minuit(
    least_squares,
    amplitude=900,
    mean=125.0,
    sigma=2.0,
)

fit.migrad()
fit.hesse()

print(fit)

print(f"Amplitude = {fit.values['amplitude']:.1f}")
print(f"Mean      = {fit.values['mean']:.2f} GeV")
print(f"Sigma     = {fit.values['sigma']:.2f} GeV")

# Plot the fitted model.
x = np.linspace(80, 170, 500)

plt.errorbar(
    centers,
    counts,
    yerr=errors,
    fmt="o",
    label="Data",
)

plt.plot(
    x,
    gaussian(
        x,
        fit.values["amplitude"],
        fit.values["mean"],
        fit.values["sigma"],
    ),
    lw=2,
    label="Gaussian fit",
)

plt.xlabel("Invariant mass [GeV]")
plt.ylabel("Events")
plt.legend()
plt.show()
```

**Discussion**

The fitted parameters correspond to:

* **Amplitude** — the height of the Gaussian peak.
* **Mean** — the estimated resonance mass.
* **Sigma** — the detector resolution (or intrinsic width, depending on the model).

In a realistic analysis, the signal would usually be fitted together with a background model rather than with a Gaussian alone.

</details>


In [ ]:
# Your solution here

# 6. Putting the tools together

A realistic HEP analysis often follows this pattern:

```text
Collision events
      │
      ▼
Read structured event data
      │
      ▼
Build particle-level observables
      │
      ▼
Cluster particles into jets
      │
      ▼
Apply event selections
      │
      ▼
Fill histograms
      │
      ▼
Fit model parameters
      │
      ▼
Perform statistical inference
      │
      ▼
Physics result
```

The packages introduced here each address a different part of this workflow:

| Package | Role |
|---|---|
| `particle` | Particle properties and PDG identifiers |
| `fastjet` | Jet clustering |
| `hist` / `mplhep` | Histogramming and HEP-style plotting |
| `iminuit` | Parameter estimation |
| `pyhf` | Likelihood-based statistical inference |

# Final mini-project: A toy resonance search

Build a simplified resonance-search workflow.

## Tasks

1. Generate or load a background mass distribution
2. Add a Gaussian signal peak
3. Fill and plot the invariant-mass histogram
4. Fit the signal peak and background with `iminuit`
5. Define signal and background yields in a signal region
6. Build a one-bin or multi-bin `pyhf` model
7. Fit the signal strength
8. Compute a hypothesis-test result

## Optional extensions

- Add a background-shape uncertainty
- Compare multiple signal hypotheses
- Scan over resonance masses
- Plot the fitted signal and background components
- Replace NumPy records with Awkward Arrays
- Read a ROOT file with Uproot

In [ ]:
# Mini-project workspace

# Summary

You have used:

- `particle` to inspect particle properties
- `fastjet` to cluster particles into jets
- `hist` and `mplhep` to visualize HEP distributions
- `iminuit` to estimate model parameters
- `pyhf` to construct and test likelihood models

These libraries complement the broader Scientific Python ecosystem and provide reusable building blocks for modern HEP analyses.